In [0]:
%run "/Workspace/Users/pjadhav564@gmail.com/Brazil_project/functions"

In [0]:
# read silver_path and write to gold_path
silver_path = "/Volumes/e_commerce_brazil/e_com_silver/updated_silver"
gold_path  =  "/Volumes/e_commerce_brazil/e_com_gold/business_ready/"

## read dim_customers

In [0]:
dim_customer = read_csv(f"{silver_path}/silver_customers")
display(dim_customer)
print(dim_customer)

In [0]:
#Gold dimension customers

dim_customer = dim_customer.select("customer_id","customer_unique_id","customer_zip_code_prefix","customer_city","customer_state").dropDuplicates(["customer_id"])    

# write dim_customers
gold_path  = "/Volumes/e_commerce_brazil/e_com_gold/business_ready/"
dim_customer.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{gold_path}/dim_customer")


## read dim_product

In [0]:
#Gold dimension products
df_clean_products = read_csv(f"{silver_path}/product_silver")
df_product_category = read_csv(f"{silver_path}/category_silver")

dim_product = df_clean_products.join(
    df_product_category, "product_category_name", "left"
).select(
    coalesce(col("product_category_name_english"),col("product_category_name")).alias(
        "product_category_name"
    ),
    "product_description_length",
    "product_height_cm",
    "product_id",
    "product_length_cm",
    "product_name_length",
    "product_photos_qty",
    "product_weight_g",
    "product_width_cm",
)

dim_product = dim_product.dropDuplicates(["product_id"])
display(dim_product)

#write dim_products
dim_customer.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{gold_path}/dim_products")

## read dim_seller 

In [0]:
#Gold dimension sellers


df_clean_sellers = read_csv(f"{silver_path}/sellers_silver")
dim_sellers= df_clean_sellers.select("seller_id","seller_zip_code_prefix","seller_city","seller_state")
dim_sellers = dim_sellers.dropDuplicates(["seller_id"])
dim_sellers = dim_sellers.select("seller_id","seller_city","seller_state","seller_zip_code_prefix")
dim_sellers = dim_sellers.dropDuplicates(["seller_id"])
dim_sellers = dim_sellers.withColumn("start_date",current_timestamp()).withColumn("end_date", lit("9999-12-31").cast("timestamp")).withColumn("is_active",lit(1))


display(dim_sellers)

#write dim_products
writefile(dim_sellers,f"{gold_path}/dim_sellers","overwrite")


## read dim_category

In [0]:
#Gold dimension category


df_clean_category = read_csv(f"{silver_path}/category_silver")

dim_category = df_clean_category.select("product_category_name","product_category_name_english")
dim_category = dim_category.dropDuplicates(["product_category_name"])

display(dim_category)

#Write dim_category
writefile(dim_category,f"{gold_path}/dim_category","overwrite")


## create dimention date table

In [0]:
# dim_date: A derived and calculated dimension that provides date-related attributes such as year, quarter, month, week, day, and day name. It is generated from order_purchase_timestamp in the Silver Orders table and is used for time-based analysis in the Gold layer.

df_clean_orders = read_csv(f"{silver_path}/orders_silver")
dim_date = df_clean_orders.select(to_date(col("order_purchase_timestamp")).alias("full_date")).distinct()\
.withColumn("date_id", date_format(col("full_date"), "yyyyMMdd").cast("int"))\
.withColumn("year", year(col("full_date")))\
.withColumn("quarter", quarter(col("full_date")))\
.withColumn("month", month(col("full_date")))\
.withColumn("month_name", date_format(col("full_date"), "MMMM"))\
.withColumn("week", weekofyear(col("full_date")))\
.withColumn("day", dayofmonth(col("full_date")))\
.withColumn("day_name", date_format(col("full_date"), "EEEE"))\
.withColumn("day_of_week", dayofweek(col("full_date")))

display(dim_date)


#Write dim_date

writefile(dim_date, f"{gold_path}/dim_date","overwrite")